# Waterfall Plot — Pressão de Combustão Cíclica & Análise de Detonação

Reproduz o gráfico `Graphic.png` a partir de `P_H2.csv`.

**Estrutura dos dados:** cada coluna do CSV (MP 74–166 → 93 colunas) é uma **geração de teste** (ciclo). As linhas são o **ângulo de virabrequim** (`-360°` a `+360°`, passo 0,1°) e os valores são a **pressão** `[bar]`.

- **Eixo X** → Ângulo de virabrequim `[deg aTDC]`
- **Eixo Y** → Ciclo (geração de teste)
- **Eixo Z** → Pressão `[bar]`

Dois gráficos interativos:
1. **Waterfall 3D** — superfície que você gira/zoom com o mouse.
2. **Navegador de gerações** — slider + play para percorrer ciclo a ciclo.

> Dependências: `pandas`, `numpy`, `plotly`. Se faltar algo, rode a célula de instalação abaixo.

In [1]:
# Execute uma vez se faltar alguma biblioteca:
# %pip install pandas numpy plotly

In [2]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# ----------------------- CONFIGURAÇÃO -----------------------
CSV_PATH = r"C:\Users\mmm\OneDrive\Projects\_Giovani\P_H2.csv"

# Janela de ângulo de virabrequim a exibir (graus aTDC).
# A combustão/detonação fica em torno do PMS (0°); a imagem usa ~ -20° a 60°.
CRANK_MIN, CRANK_MAX = -30.0, 70.0

# Subamostragem do ângulo: 1 = todos os pontos, 2 = metade, 4 = 1/4 ...
# Valores maiores deixam o 3D mais leve.
ANGLE_STEP = 2
# ------------------------------------------------------------

In [3]:
# Linha 0 do CSV = números MP (rótulos das colunas/ciclos). Dados começam na linha 4.
mp_labels = pd.read_csv(CSV_PATH, sep=';', header=None, nrows=1).iloc[0, 1:].tolist()
raw = pd.read_csv(CSV_PATH, sep=';', header=None, skiprows=3, decimal='.')

crank_full = raw.iloc[:, 0].to_numpy(dtype=float)      # ângulo de virabrequim
P_full = raw.iloc[:, 1:].to_numpy(dtype=float)         # (n_angulos, n_ciclos)
n_cycles = P_full.shape[1]
cycles = np.arange(1, n_cycles + 1)                    # 1..93

print(f"{P_full.shape[0]} ângulos x {n_cycles} ciclos carregados")
print(f"Pressão: {np.nanmin(P_full):.2f} .. {np.nanmax(P_full):.2f} bar")

7200 ângulos x 93 ciclos carregados
Pressão: 0.56 .. 71.88 bar


In [4]:
# Aplica a janela de ângulo + subamostragem
mask = (crank_full >= CRANK_MIN) & (crank_full <= CRANK_MAX)
crank = crank_full[mask][::ANGLE_STEP]
P = P_full[mask][::ANGLE_STEP, :]                      # (n_ang_janela, n_ciclos)

print(f"Janela: {crank.min():.1f}° .. {crank.max():.1f}°  ->  {crank.size} pontos x {n_cycles} ciclos")

Janela: -30.0° .. 70.0°  ->  501 pontos x 93 ciclos


## 1) Waterfall 3D interativo (superfície)

Gire arrastando com o mouse, dê zoom com a roda e passe o cursor para ler os valores. Os picos altos (vermelho) correspondem ao início de detonação (*knock onset*).

Agora também há um **slider** (e botão ▶ Play) que destaca, com uma **linha preta sobre a superfície**, a geração de teste selecionada — para navegar ciclo a ciclo dentro do próprio 3D.

In [5]:
# Superfície (traço 0) — fica fixa
surface = go.Surface(
    x=crank,            # eixo X = ângulo
    y=cycles,           # eixo Y = ciclo
    z=P.T,              # Z[ciclo, ângulo] -> shape (n_ciclos, n_angulos)
    colorscale='Jet',
    colorbar=dict(title='Pressão [bar]'),
    contours={"z": {"show": True, "usecolormap": True, "width": 1, "project": {"z": False}}},
    hovertemplate='Ângulo %{x:.1f}°<br>Ciclo %{y}<br>Pressão %{z:.1f} bar<extra></extra>',
)

zlift = 0.03 * np.nanmax(P)   # pequena elevação para a linha "flutuar" sobre a superfície

def highlight_line(j):
    """Linha 3D destacando o ciclo j (índice 0-based)."""
    return go.Scatter3d(
        x=crank,
        y=np.full(crank.shape, cycles[j]),
        z=P[:, j] + zlift,
        mode='lines',
        line=dict(color='black', width=6),
        name='Ciclo selecionado',
        hovertemplate=f'Ciclo {cycles[j]}<br>Ângulo %{{x:.1f}}°<br>Pressão %{{z:.1f}} bar<extra></extra>',
    )

fig3d = go.Figure(data=[surface, highlight_line(0)])

# Um frame por ciclo: atualiza só a linha destacada (traço 1)
fig3d.frames = [
    go.Frame(name=str(c), traces=[1], data=[highlight_line(j)])
    for j, c in enumerate(cycles)
]

steps = [dict(method='animate', label=str(c),
              args=[[str(c)], dict(mode='immediate',
                                   frame=dict(duration=0, redraw=True),
                                   transition=dict(duration=0))])
         for c in cycles]

fig3d.update_layout(
    title='Waterfall 3D: Pressão de Combustão Cíclica & Análise de Detonação',
    scene=dict(
        xaxis_title='Ângulo de virabrequim [deg aTDC]',
        yaxis_title='Ciclo (geração)',
        zaxis_title='Pressão [bar]',
        camera=dict(eye=dict(x=1.8, y=-1.7, z=0.8)),
        aspectratio=dict(x=1.4, y=1.0, z=0.6),
    ),
    width=1000, height=720, margin=dict(l=0, r=0, t=50, b=0),
    sliders=[dict(active=0, currentvalue=dict(prefix='Ciclo (geração): '),
                  pad=dict(t=20, b=10), len=0.9, x=0.05, steps=steps)],
    updatemenus=[dict(type='buttons', direction='left', showactive=False,
                      x=0.05, y=0, xanchor='left', yanchor='top',
                      buttons=[
                          dict(label='▶ Play', method='animate',
                               args=[None, dict(frame=dict(duration=150, redraw=True),
                                                fromcurrent=True, transition=dict(duration=0))]),
                          dict(label='⏸ Pause', method='animate',
                               args=[[None], dict(mode='immediate',
                                                  frame=dict(duration=0, redraw=True))]),
                      ])],
)
# Gire/zoom com o mouse; use o slider ou ▶ Play para destacar cada geração.
fig3d.show()

## 2) Navegador de gerações de teste

Use o **slider** (ou o botão ▶ Play) para percorrer os ciclos um a um. O ciclo selecionado aparece em **vermelho** sobre o fundo esmaecido com todos os ciclos — assim dá para ver onde aquela geração se desvia das demais (ex.: detonação).

In [6]:
# Fundo: todos os ciclos esmaecidos
bg = [go.Scatter(x=crank, y=P[:, j], mode='lines',
                 line=dict(color='lightgray', width=0.5),
                 hoverinfo='skip', showlegend=False)
      for j in range(n_cycles)]

# Traço destacado (ciclo selecionado) — inicia no ciclo 1
hi = go.Scatter(x=crank, y=P[:, 0], mode='lines',
                line=dict(color='crimson', width=2.5),
                name='Ciclo selecionado')

fig2d = go.Figure(data=bg + [hi])
hi_idx = len(bg)  # índice do traço que será atualizado pelo slider

# Um frame por ciclo: atualiza só o traço destacado
fig2d.frames = [
    go.Frame(name=str(c), traces=[hi_idx],
             data=[go.Scatter(x=crank, y=P[:, j])])
    for j, c in enumerate(cycles)
]

steps = [dict(method='animate', label=str(c),
              args=[[str(c)], dict(mode='immediate',
                                   frame=dict(duration=0, redraw=True),
                                   transition=dict(duration=0))])
         for c in cycles]

fig2d.update_layout(
    title='Navegador de gerações de teste',
    xaxis_title='Ângulo de virabrequim [deg aTDC]',
    yaxis_title='Pressão [bar]',
    sliders=[dict(active=0, currentvalue=dict(prefix='Ciclo (geração): '),
                  pad=dict(t=40), steps=steps)],
    updatemenus=[dict(type='buttons', direction='left', showactive=False,
                      x=0.0, y=1.12, xanchor='left', yanchor='top',
                      buttons=[
                          dict(label='▶ Play', method='animate',
                               args=[None, dict(frame=dict(duration=120, redraw=True),
                                                fromcurrent=True, transition=dict(duration=0))]),
                          dict(label='⏸ Pause', method='animate',
                               args=[[None], dict(mode='immediate',
                                                  frame=dict(duration=0, redraw=True))]),
                      ])],
    width=1000, height=600, margin=dict(t=80),
)
fig2d.show()

---
### Dicas
- Para ver **toda** a faixa de ângulo, ajuste `CRANK_MIN`/`CRANK_MAX` (ex.: `-180`, `180`) e reexecute as células.
- Se o 3D ficar pesado, aumente `ANGLE_STEP` (ex.: `4` ou `8`).
- Para exportar um dos gráficos como HTML interativo: `fig3d.write_html('waterfall.html')`.